# Parameter-Efficient Fine-Tuning of TinyLlama with LoRA

This project fine-tunes **TinyLlama-1.1B-Chat** on 1,000 examples from the **Databricks Dolly 15K** instruction dataset using supervised fine-tuning and Low-Rank Adaptation (LoRA).

The goal is to improve instruction following while updating only a small fraction of the model. The completed run trained **1,126,400 parameters (0.1023%)**, produced a **4.31 MB** adapter, and reduced logged training loss from **2.1047** at step 10 to **1.7223** at step 60.

## Project highlights

- Fine-tunes a 1.1-billion-parameter language model on a 4 GB laptop GPU.
- Converts Dolly records into role-based chat conversations.
- Applies LoRA only to the query and value attention projections.
- Uses gradient accumulation and checkpointing to control memory use.
- Compares responses before and after fine-tuning.
- Saves a reusable PEFT adapter instead of a full model copy.


## 1 Setup


## Workflow

1. Configure the experiment and load the model.
2. Generate baseline responses for a small qualitative benchmark.
3. Convert Dolly records into user–assistant conversations.
4. Insert LoRA adapters into the attention layers.
5. Train for one epoch with an effective batch size of 16.
6. Save the adapter and tokenizer.
7. Compare the original and fine-tuned responses.

The final run completed **63 optimizer steps** in approximately **465 seconds**. Its average training loss was **1.8282**, and mean token accuracy was **0.6167**.


In [1]:
%pip install -U transformers trl peft datasets accelerate ipywidgets -q


Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import gc
import os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.7.1+cu126
CUDA available: True


GPU: NVIDIA GeForce RTX 3050 Laptop GPU


### Training and evaluation helpers

These functions release GPU memory, report the number of trainable parameters, and generate chat responses consistently for the baseline and fine-tuned models.


In [3]:
def clear_memory():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


def print_trainable_params(model):
    """Display trainable vs total parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:>12,}  ({100 * trainable / total:.4f}%)")
    print(f"Total parameters:     {total:>12,}")


def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate a response from a model given a prompt string."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response


---
## 2 Experiment configuration

All model, dataset, LoRA, and optimization settings are defined in one place. A micro-batch of 1 with 16 gradient-accumulation steps preserves an effective batch size of 16 while fitting within 4 GB of VRAM.


### Configuration rationale

The LoRA rank controls adapter capacity, `lora_alpha` scales the adapter update, and dropout regularizes the LoRA path. Targeting `q_proj` and `v_proj` limits updates to the query and value projections in self-attention. The fixed random seed makes data selection and sampled comparisons more reproducible.


In [4]:
# Central experiment configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "databricks/databricks-dolly-15k"

NUM_TRAIN_SAMPLES = 1000
MAX_SEQ_LENGTH = 512

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

LEARNING_RATE = 2e-4
NUM_EPOCHS = 1

# A micro-batch of 1 and 16 accumulation steps fit within 4 GB of VRAM
# while preserving an effective batch size of 16.
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16

OUTPUT_DIR = "./sft-lora-tinyllama"

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)
print("Training samples:", NUM_TRAIN_SAMPLES)
print("Maximum sequence length:", MAX_SEQ_LENGTH)
print("Micro-batch size:", BATCH_SIZE)
print("Gradient accumulation steps:", GRAD_ACCUM_STEPS)
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM_STEPS)


Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Dataset: databricks/databricks-dolly-15k
Training samples: 1000
Maximum sequence length: 512
Micro-batch size: 1
Gradient accumulation steps: 16
Effective batch size: 16


---
## 3 Load the tokenizer and base model

The tokenizer and TinyLlama base model are loaded from Hugging Face. The model uses half precision and automatic device placement to reduce memory use.


### Step explanation

The tokenizer converts text to token IDs and back. TinyLlama does not define a separate padding token, so I reuse the end-of-sequence token for padding. The attention mask prevents padded positions from contributing to the calculation. The model is loaded in FP16 to reduce the memory required for its base weights, and `device_map="auto"` places it on the available hardware.

### Observation

The model loaded correctly on the NVIDIA RTX 3050 and contains **1,100,048,384 parameters**, using about **2.05 GB** of allocated GPU memory. The installed Transformers version warns that `torch_dtype` is deprecated in favor of `dtype`; this warning does not affect the completed run.


In [5]:
# load tokenizer and model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Select the CUDA-enabled Python kernel "
        "and restart the notebook."
    )

print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# TinyLlama does not normally define a separate padding token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

total_parameters = sum(parameter.numel() for parameter in model.parameters())

print(f"Model device: {model.device}")
print(f"Model precision: {model.dtype}")
print(f"Total parameters: {total_parameters:,}")
print(
    f"GPU memory allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)


GPU: NVIDIA GeForce RTX 3050 Laptop GPU


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model device: cuda:0
Model precision: torch.float16
Total parameters: 1,100,048,384
GPU memory allocated: 2.05 GB


---
## 4 Establish a baseline

Before training, the base model answers two prompts that test concise explanation and list formatting. These responses are stored for a direct comparison after fine-tuning.


### Step explanation

These generations establish a baseline before training. Saving them is essential because it lets us compare the same prompts after fine-tuning. Fixing the random seed improves reproducibility, although sampled GPU generation can still vary slightly across software or hardware versions.

### Observation

The base model answered both prompts sensibly, but the first answer exceeded the requested 2–3 sentences and was repetitive. The second answer was cut off because generation reached the 128-token limit. These are useful baseline weaknesses to compare against the tuned model.


In [6]:
# define test_prompts, generate base model responses, store in base_responses
test_prompts = [
    "Explain what a neural network is in 2-3 sentences.",
    "What are three tips for better sleep?",
]

base_responses = []

# Reset the random seed so results can be reproduced.
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

for prompt in test_prompts:
    response = generate_response(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=128,
    )

    base_responses.append(response)

    print("=" * 80)
    print("PROMPT:")
    print(prompt)
    print("\nBASE MODEL RESPONSE:")
    print(response)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
Explain what a neural network is in 2-3 sentences.

BASE MODEL RESPONSE:
A neural network is a mathematical model that mimics the structure and function of the human brain. It consists of a set of interconnected units, or neurons, that receive input from other neurons through synapses, process the input, and output a response. Neural networks are trained using data to learn complex functions and solve problems. The output of a neural network is a prediction or output that can be used to make decisions or make predictions.


PROMPT:
What are three tips for better sleep?

BASE MODEL RESPONSE:
Sure, here are three tips for better sleep:

1. Create a sleep-conducive environment: Make sure your bedroom is dark, cool, and quiet. Keep your bedroom temperature around 60-67°F (15-20°C) and avoid using electronics or other distractions before bedtime.

2. Establish a regular sleep schedule: Set a regular sleep-wake schedule, and stick to it. Avoid going to bed or waking up during the day.

3. Avoid caffeine, alcohol,


In [7]:
assert len(base_responses) == len(test_prompts)
assert all(isinstance(response, str) for response in base_responses)

print(f"Saved {len(base_responses)} baseline responses.")


Saved 2 baseline responses.


In [8]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


---
## 5 Prepare the Dolly instruction dataset

Each Dolly record is converted from separate `instruction`, `context`, and `response` fields into a two-message chat conversation. The dataset is shuffled with a fixed seed before selecting 1,000 training examples.


### Dataset check

The full dataset contains **15,011 rows**. I shuffled it with a fixed seed, selected **1,000 rows**, removed the original columns after formatting, and checked the two-message structure with assertions. A blank line separates the instruction from its context when context is available.


### 5.1 Load and inspect Dolly 15K


In [9]:
# load dolly dataset, write format_dolly_to_chat, apply it
from datasets import load_dataset

raw_dataset = load_dataset(
    DATASET_NAME,
    split="train",
)

print(raw_dataset)
print(f"Number of examples: {len(raw_dataset):,}")
print("Columns:", raw_dataset.column_names)
print("\nFirst raw example:")

for column, value in raw_dataset[0].items():
    print(f"{column}: {value!r}")


Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})
Number of examples: 15,011
Columns: ['instruction', 'context', 'response', 'category']

First raw example:
instruction: 'When did Virgin Australia start operating?'
context: "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."
response: 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.'
category: 'closed_qa'


### 5.2 Convert records to chat format


In [10]:
def format_dolly_to_chat(example):
    instruction = (example.get("instruction") or "").strip()
    context = (example.get("context") or "").strip()
    response = (example.get("response") or "").strip()

    # Add context only when it contains meaningful text.
    if context:
        user_content = f"{instruction}\n\n{context}"
    else:
        user_content = instruction

    return {
        "messages": [
            {
                "role": "user",
                "content": user_content,
            },
            {
                "role": "assistant",
                "content": response,
            },
        ]
    }


In [11]:
example_without_context = {
    "instruction": "What is machine learning?",
    "context": "",
    "response": "Machine learning allows systems to learn patterns from data.",
}

example_with_context = {
    "instruction": "Summarize the following passage.",
    "context": "Neural networks are composed of interconnected layers.",
    "response": "Neural networks contain interconnected layers.",
}

print(format_dolly_to_chat(example_without_context))
print()
print(format_dolly_to_chat(example_with_context))


{'messages': [{'role': 'user', 'content': 'What is machine learning?'}, {'role': 'assistant', 'content': 'Machine learning allows systems to learn patterns from data.'}]}

{'messages': [{'role': 'user', 'content': 'Summarize the following passage.\n\nNeural networks are composed of interconnected layers.'}, {'role': 'assistant', 'content': 'Neural networks contain interconnected layers.'}]}


### 5.3 Shuffle and select the training subset


In [12]:
num_samples = min(NUM_TRAIN_SAMPLES, len(raw_dataset))

training_subset = (
    raw_dataset
    .shuffle(seed=42)
    .select(range(num_samples))
)

formatted_dataset = training_subset.map(
    format_dolly_to_chat,
    remove_columns=training_subset.column_names,
)

print(f"Raw dataset size: {len(raw_dataset):,}")
print(f"Training subset size: {len(formatted_dataset):,}")
print("Formatted columns:", formatted_dataset.column_names)


Raw dataset size: 15,011
Training subset size: 1,000
Formatted columns: ['messages']


In [13]:
formatted_example = formatted_dataset[0]

print("\nFormatted example:")
print(formatted_example)

assert "messages" in formatted_example
assert len(formatted_example["messages"]) == 2
assert formatted_example["messages"][0]["role"] == "user"
assert formatted_example["messages"][1]["role"] == "assistant"

print("\nDataset structure is valid.")



Formatted example:
{'messages': [{'role': 'user', 'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?'}, {'role': 'assistant', 'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree'}]}

Dataset structure is valid.


In [14]:
for message in formatted_dataset[0]["messages"]:
    print(f"{message['role'].upper()}:")
    print(message["content"])
    print()


USER:
Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?

ASSISTANT:
Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree



---
## 6 Configure LoRA

Low-rank adapters are inserted into TinyLlama's query and value projections. The base weights remain frozen, so optimization updates only the small adapter matrices.


### Step explanation

LoRA represents a weight update with two small low-rank matrices rather than modifying the full projection matrix. Here, rank 8 adapters are inserted into every query and value projection. The scaling factor is `alpha / rank = 16 / 8 = 2`, and dropout of 0.05 is applied on the LoRA path during training.

### Parameter check

The printed layer shows that the original 2048×2048 linear layer remains present and receives `lora_A` and `lora_B` modules. Only **1,126,400 of 1,101,174,784 parameters**, or **0.1023%**, are trainable. The final assertion checks that no non-LoRA parameter is unexpectedly trainable.


### 6.1 Define the LoRA configuration


In [15]:
# create LoraConfig, apply to model, print trainable params

from peft import LoraConfig, TaskType


lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
)

print(lora_config)


LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


### 6.2 Attach adapters to the base model


In [16]:
from peft import get_peft_model

peft_model = get_peft_model(
    model,
    lora_config,
)

print("LoRA adapters successfully added.")


LoRA adapters successfully added.


In [17]:
print(peft_model.base_model.model.model.layers[0].self_attn.q_proj)


lora.Linear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=2048, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=2048, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


### 6.3 Verify trainable parameters


In [18]:
peft_model.print_trainable_parameters()


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [19]:
trainable_parameters = sum(
    parameter.numel()
    for parameter in peft_model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in peft_model.parameters()
)

trainable_percentage = (
    100 * trainable_parameters / total_parameters
)

print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Total parameters:     {total_parameters:,}")
print(f"Trainable percentage: {trainable_percentage:.4f}%")


Trainable parameters: 1,126,400
Total parameters:     1,101,174,784
Trainable percentage: 0.1023%


In [20]:
assert trainable_parameters > 0, "No trainable LoRA parameters were found."
assert trainable_percentage < 1.0, (
    f"Expected under 1% trainable parameters, "
    f"but found {trainable_percentage:.4f}%."
)

print("Validation passed: less than 1% of parameters are trainable.")


Validation passed: less than 1% of parameters are trainable.


In [21]:
unexpected_trainable_parameters = [
    name
    for name, parameter in peft_model.named_parameters()
    if parameter.requires_grad and "lora_" not in name
]

print(
    "Unexpected trainable parameters:",
    unexpected_trainable_parameters,
)

assert not unexpected_trainable_parameters


Unexpected trainable parameters: []


---
## 7 Supervised fine-tuning

`SFTTrainer` tokenizes the chat conversations, truncates sequences to 512 tokens, and performs next-token prediction. Gradient checkpointing and a one-example micro-batch keep memory usage within the laptop GPU limit.


### Step explanation

`SFTTrainer` tokenizes the chat records, truncates them to 512 tokens, computes next-token prediction loss, and updates only the adapter weights. Gradient accumulation combines several micro-batches before one optimizer update, while checkpointing saves memory by recomputing intermediate activations during backpropagation. The cosine scheduler gradually lowers the learning rate after warm-up.

### Implementation choices

- A micro-batch of 1 and 16 gradient-accumulation steps provide an effective batch size of 16 while limiting peak VRAM use.
- The PEFT-wrapped model is passed directly to `SFTTrainer`, ensuring that the existing LoRA adapters are optimized without adding a second adapter set.
- Training precision is selected automatically: bf16 when the active CUDA environment reports support, otherwise fp16.

### Observation

Training completed for one epoch in about **465 seconds** and produced **63 optimizer steps**. The average epoch loss was **1.8282**, while the logged loss decreased from **2.1047** at step 10 to **1.7223** at step 60. Loss dropped quickly over the first 30 steps and then plateaued around 1.72–1.78, which is a healthy, stable curve for one epoch on 1,000 examples; it is neither flat nor erratic. Mean token accuracy was **0.6167**, and the run completed without an error.


### Training limitation

In this configuration, loss is calculated across the full formatted conversation, including the user prompt. Setting `assistant_only_loss=True` would restrict the objective to assistant tokens, provided the chat template supplies the required assistant-token mask. I kept the default full-conversation loss so the run remains aligned with the project configuration.


### 7.1 Configure and run training


In [23]:
import math
from trl import SFTConfig

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()


# Approximately 63 optimizer steps for 1,000 examples.
steps_per_epoch = math.ceil(
    len(formatted_dataset) /
    (BATCH_SIZE * GRAD_ACCUM_STEPS)
)

total_training_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(total_training_steps * 0.03))

print("Estimated optimizer steps:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("Training precision:", "bf16" if use_bf16 else "fp16")

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    optim="adamw_torch",

    bf16=use_bf16,
    fp16=not use_bf16,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    max_length=MAX_SEQ_LENGTH,

    logging_strategy="steps",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",

    packing=False,
    seed=42,
)

print("Training configuration created successfully.")


Estimated optimizer steps: 63
Warmup steps: 1
Training precision: bf16
Training configuration created successfully.


In [24]:
peft_model.config.use_cache = False

trainer = SFTTrainer(
    model=peft_model,
    args=training_config,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
)

print("Trainer created successfully.")


Trainer created successfully.


In [25]:
train_result = trainer.train()
print(train_result.metrics)


Step,Training Loss
10,2.104679
20,1.874144
30,1.757696
40,1.734372
50,1.780185
60,1.722342


{'train_runtime': 464.9696, 'train_samples_per_second': 2.151, 'train_steps_per_second': 0.135, 'total_flos': 1191545917820928.0, 'train_loss': 1.8282204355512346, 'entropy': 1.7385170832276344, 'mean_token_accuracy': 0.6167212665081024, 'num_tokens': 191757.0, 'epoch': 1.0}


---
## 8 Save and inspect the adapter

The trained PEFT model saves only the LoRA configuration and adapter weights, together with the tokenizer files required for inference.


### Step explanation

Saving a PEFT model writes only the learned adapter configuration and weights, not another 2.2 GB copy of TinyLlama. Saving the tokenizer alongside the adapter records the matching text-processing and chat-template files needed for later inference.

### Saved result

Six files were saved. The complete directory is **7.77 MB**, of which the actual LoRA weights occupy **4.31 MB**; most of the remaining size is the tokenizer. This compact result is the main storage advantage of LoRA.


### 8.1 Save the adapter and tokenizer


In [26]:
# save adapter and tokenizer, print adapter size
from pathlib import Path

adapter_path = Path(OUTPUT_DIR) / "final-adapter"
adapter_path.mkdir(parents=True, exist_ok=True)

# trainer.model is a PEFT model, so this saves the adapter weights,
# not another complete copy of TinyLlama.
trainer.model.save_pretrained(
    adapter_path,
    safe_serialization=True,
)

tokenizer.save_pretrained(adapter_path)

print("Adapter and tokenizer saved to:")
print(adapter_path.resolve())


Adapter and tokenizer saved to:
D:\scaler\AI_Infusion\Lora_\sft-lora-tinyllama\final-adapter


In [27]:
print("\nSaved files:")

for file_path in sorted(adapter_path.rglob("*")):
    if file_path.is_file():
        print("-", file_path.name)



Saved files:
- adapter_config.json
- adapter_model.safetensors
- chat_template.jinja
- README.md
- tokenizer.json
- tokenizer_config.json


### 8.2 Measure the saved artifact


In [28]:
saved_files = [
    file_path
    for file_path in adapter_path.rglob("*")
    if file_path.is_file()
]

total_size_bytes = sum(
    file_path.stat().st_size
    for file_path in saved_files
)

total_size_mb = total_size_bytes / (1024 ** 2)

print(f"Number of saved files: {len(saved_files)}")
print(f"Total saved size: {total_size_mb:.2f} MB")


Number of saved files: 6
Total saved size: 7.77 MB


In [29]:
print("\nFile sizes:")

for file_path in sorted(saved_files):
    file_size_mb = file_path.stat().st_size / (1024 ** 2)
    relative_path = file_path.relative_to(adapter_path)

    print(f"{str(relative_path):45} {file_size_mb:8.2f} MB")



File sizes:
adapter_config.json                               0.00 MB
adapter_model.safetensors                         4.31 MB
chat_template.jinja                               0.00 MB
README.md                                         0.00 MB
tokenizer.json                                    3.45 MB
tokenizer_config.json                             0.00 MB


In [30]:
adapter_weight_files = list(
    adapter_path.glob("adapter_model.*")
)

adapter_weights_size = sum(
    file_path.stat().st_size
    for file_path in adapter_weight_files
)

print(
    f"\nAdapter weights only: "
    f"{adapter_weights_size / (1024 ** 2):.2f} MB"
)



Adapter weights only: 4.31 MB


### 8.3 Generate fine-tuned responses


### Step explanation

The trained adapter remains attached to `trainer.model`, so it can be evaluated directly. The same prompts and random seed are reused to make the qualitative comparison as fair as possible.

### Observation

For the neural-network prompt, the tuned response is exactly two sentences and removes the baseline's repetition. For the sleep prompt, it supplies three complete numbered tips, whereas the baseline was truncated; however, it also adds a redundant closing sentence. This suggests improved instruction and format following on these examples. It does **not** prove broad quality improvement: two sampled prompts are only a qualitative check, and a stronger evaluation would use a held-out dataset plus task-specific or human-rated metrics.


In [31]:
import torch

fine_tuned_model = trainer.model

fine_tuned_model.config.use_cache = True
fine_tuned_model.eval()

print("Fine-tuned model ready for evaluation.")


Fine-tuned model ready for evaluation.


In [32]:
assert len(test_prompts) == len(base_responses)
assert len(test_prompts) > 0

print(f"Evaluating {len(test_prompts)} prompts.")


Evaluating 2 prompts.


In [33]:
fine_tuned_responses = []

# Use the same seed used for baseline generation.
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

for prompt in test_prompts:
    response = generate_response(
        model=fine_tuned_model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=128,
    )

    fine_tuned_responses.append(response)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [34]:
assert len(fine_tuned_responses) == len(test_prompts)

print(
    f"Generated {len(fine_tuned_responses)} "
    "fine-tuned responses."
)


Generated 2 fine-tuned responses.


In [35]:
for index, (prompt, baseline, fine_tuned) in enumerate(
    zip(
        test_prompts,
        base_responses,
        fine_tuned_responses,
    ),
    start=1,
):
    print("=" * 100)
    print(f"TEST {index}")
    print("=" * 100)

    print("\nPROMPT:")
    print(prompt)

    print("\nBASE MODEL RESPONSE:")
    print(baseline)

    print("\nFINE-TUNED MODEL RESPONSE:")
    print(fine_tuned)

    print()


TEST 1

PROMPT:
Explain what a neural network is in 2-3 sentences.

BASE MODEL RESPONSE:
A neural network is a mathematical model that mimics the structure and function of the human brain. It consists of a set of interconnected units, or neurons, that receive input from other neurons through synapses, process the input, and output a response. Neural networks are trained using data to learn complex functions and solve problems. The output of a neural network is a prediction or output that can be used to make decisions or make predictions.

FINE-TUNED MODEL RESPONSE:
A neural network is a mathematical model that mimics the structure and function of the human brain. It consists of a set of layers, each with a different function, connected by weights and biases.

TEST 2

PROMPT:
What are three tips for better sleep?

BASE MODEL RESPONSE:
Sure, here are three tips for better sleep:

1. Create a sleep-conducive environment: Make sure your bedroom is dark, cool, and quiet. Keep your bedroom t

In [36]:
import pandas as pd

comparison = pd.DataFrame({
    "Prompt": test_prompts,
    "Baseline response": base_responses,
    "Fine-tuned response": fine_tuned_responses,
})

display(comparison)


,Prompt,Baseline response,Fine-tuned response
0,Explain what a neural network is in 2-3 senten...,A neural network is a mathematical model that ...,A neural network is a mathematical model that ...
1,What are three tips for better sleep?,"Sure, here are three tips for better sleep:\n\...",1. Limit caffeine and alcohol before bed.\n2. ...


---
## 9 Evaluate response quality

The same prompts and generation seed are used for the base and fine-tuned models. This small comparison focuses on instruction compliance, completeness, clarity, factuality, and repetition.


### Evaluation guidance

I compare instruction compliance, completeness, clarity, factuality, and unnecessary repetition—not only training loss. The displayed table provides a direct qualitative comparison. Because generation uses sampling, these conclusions should ideally be checked across more prompts or with deterministic decoding. The displayed responses were produced by the final end-to-end run; the fixed seed makes repeated text possible even when training precision changes slightly.

### Overall conclusion

The experiment produced a compact LoRA adapter and improved format adherence on both test prompts. The neural-network answer became shorter and followed the requested sentence count. The sleep answer became complete and numbered, although its final sentence was redundant. These two examples are encouraging qualitative evidence, but they are not a substitute for evaluation on a larger held-out prompt set.


In [37]:
# The responses were generated in Part 7 and are reused here for evaluation.
assert len(test_prompts) == len(base_responses) == len(fine_tuned_responses)

comparison = pd.DataFrame({
    "Prompt": test_prompts,
    "Base model": base_responses,
    "Fine-tuned model": fine_tuned_responses,
})

display(comparison)


,Prompt,Base model,Fine-tuned model
0,Explain what a neural network is in 2-3 senten...,A neural network is a mathematical model that ...,A neural network is a mathematical model that ...
1,What are three tips for better sleep?,"Sure, here are three tips for better sleep:\n\...",1. Limit caffeine and alcohol before bed.\n2. ...


---
## Done

You have successfully fine-tuned TinyLlama with LoRA on the Dolly dataset. Review your base vs fine-tuned outputs and note the differences in instruction-following quality.


---
## Experiment summary

### Results

- The Dolly dataset was converted into valid user–assistant conversations and checked with assertions.
- LoRA adapters were attached to the query and value projections.
- Only **1,126,400 parameters (0.1023%)** were trainable.
- One epoch completed successfully with an average training loss of **1.8282**.
- The saved LoRA weights occupy **4.31 MB**.
- Both sample responses followed the requested format better after fine-tuning, though the sleep response retained a redundant closing sentence.

### Implementation choices

- The run used a micro-batch of 1 with 16 accumulation steps instead of 4 × 4. The effective batch size is still 16, and this adjustment reduces memory pressure on a 4 GB GPU.
- The trainer received the pre-wrapped `peft_model` rather than reloading the base model with `peft_config`. The resulting trainer still optimized the same LoRA parameters.
- Precision is chosen from hardware support: bf16 when supported, otherwise fp16.

### Takeaway

LoRA adapted a 1.1-billion-parameter model while training roughly one-tenth of one percent of its parameters. This made training practical on a laptop GPU and produced a small, reusable adapter.
